Method: https://proghead231.github.io/agricultural-land-abandonment-nepal/
1. Data acquisition and preprocessing
2. Create agricultural land (AL) objects
3. Calculate annual AL probability
4. Apply landTrendr to the time series of AL probabilities
5. Identify structural breaks and classify eac AL objects to abandoned (AAL), fallow (FAL) or recultivate (RAL)
6. Validate results

## Step 1: Create AL objects
Given the uncertainity in the AL classification of the land cover datasets by FRTC I will do segmentation using some algo
1. Get landsat-SR images from 2000, 2010 and 2020, preprocess them and stack them 
2. Get texture metrics from the one image, combine the metrics into one texture image and add to the stack (Using GLCM metrics here)
3. Perform segmentation and check

In [53]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
from helpers import config
from helpers import utils
Map = geemap.Map()
import os

loaded config!
loaded utils!


In [54]:
config.ROI = config.test_geo

In [55]:
dem = ee.Image("USGS/SRTMGL1_003").clip(config.ROI).rename("z")
terrain = ee.Terrain.products(dem)

l_seg_bands= ee.List(["g", "r", "nir", "swir1", "swir2", "nir_shade", "nir_savg", "ndvi"])
terrain_bands = ee.List(["slope", "northness", "eastness"])

def add_ndvi(image):
    ndvi = image.normalizedDifference(["nir", "r"]).rename("ndvi")
    return image.addBands(ndvi)
# def add_ndvi(image):
#     ndvi = image.normalizedDifference(["nir", "r"]).rename("ndvi")
#     ndmi = image.normalizedDifference(["nir", "swir1"]).rename("ndmi")
#     return image.addBands([ndvi, ndmi])

l7_2000_image = utils.get_processed_landsat_collection("LANDSAT/LE07/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2000"], utils.mask_clouds_landsat75, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)
l7_2000_image = add_ndvi(l7_2000_image)
high_val_2000 = ee.Number(utils.calc_image_stats(l7_2000_image, config.ROI, config.SCALE).get("high"))
l7_2000_image = utils.add_scaled_glcm(l7_2000_image, config.ROI, config.SCALE, high_val_2000).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))
terrain_2000 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2000).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))


l5_2010_image = utils.get_processed_landsat_collection("LANDSAT/LT05/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2010"], utils.mask_clouds_landsat75, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)
l5_2010_image = add_ndvi(l5_2010_image)
high_val_2010 = ee.Number(utils.calc_image_stats(l5_2010_image, config.ROI, config.SCALE).get("high"))
l5_2010_image = utils.add_scaled_glcm(l5_2010_image, config.ROI, config.SCALE, high_val_2010).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))
# terrain_2010 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2010).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))

l8_2020_image = utils.get_processed_landsat_collection("LANDSAT/LC08/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2020"], utils.mask_clouds_landsat8, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L8_ORIGINAL_BAND_NAMES).rename(config.L8_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)
l8_2020_image = add_ndvi(l8_2020_image)
high_val_2020 = ee.Number(utils.calc_image_stats(l8_2020_image, config.ROI, config.SCALE).get("high"))
l8_2020_image = utils.add_scaled_glcm(l8_2020_image, config.ROI, config.SCALE, high_val_2020).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2020")))
# terrain_2020 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2020).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2020")))

seg_image_stack = l7_2000_image.addBands(l5_2010_image).addBands(l8_2020_image).addBands(terrain_2000)#.addBands(terrain_2010).addBands(terrain_2020)


In [74]:
seg_image_stack.reproject(crs="EPSG:32645", scale=30).reduceRegion(ee.Reducer.minMax(), config.ROI, 30, maxPixels=21308891)

In [58]:
target_crs = "EPSG:32645"
target_scale = 30
#s-10_n-20_cn-8_cm-1
seeds = ee.Algorithms.Image.Segmentation.seedGrid(20, "hex").reproject(crs=target_crs, scale=target_scale)
snic = ee.Algorithms.Image.Segmentation.SNIC(
  image= seg_image_stack.reproject(crs=target_crs, scale=target_scale),
  connectivity= 8,
  neighborhoodSize= 20,
  seeds= seeds,
  compactness=0.1
)
contours = snic.select('clusters') \
    .reduceToVectors(geometry=config.ROI, scale=30, maxPixels=1e13)

In [59]:
def get_s2_reference(roi, date_range):
    # 1. Define Sentinel-2 Cloud Mask
    def mask_s2_clouds(image):
        qa = image.select('QA60')
        # Bits 10 and 11 are clouds and cirrus, respectively.
        cloudBitMask = 1 << 10
        cirrusBitMask = 1 << 11
        mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
            .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
        return image.updateMask(mask).divide(10000) # Apply scaling (0.0001) here

    # 2. Fetch Collection
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(roi) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
        .map(mask_s2_clouds) \
        .median() \
        .clip(roi)

    return s2

s2_dates = ["2020-01-01", "2021-12-28"]

# Get the image
s2_2020_image = get_s2_reference(config.ROI, s2_dates)

vis_params_s2 = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}
Map.addLayer(s2_2020_image, vis_params_s2, 'Sentinel-2 (10m) Ref', False)

vis_params = {"min":0, "max":0.3, "bands": ["nir", "swir1", "r"]} 

Map.addLayer(ee.ImageCollection([l5_2010_image.select(["g_2010", "r_2010", "nir_2010", "swir1_2010"]).rename(["g", "r", "nir", "swir1"]),l7_2000_image.select(["g_2000", "r_2000", "nir_2000", "swir1_2000"]).rename(["g", "r", "nir", "swir1"]),l8_2020_image.select(["g_2020", "r_2020", "nir_2020", "swir1_2020"]).rename(["g", "r", "nir", "swir1"])]).mean(), vis_params, "l8")
# Map.addLayer(seg_image_stack.select("nir_savg_2020"), {"min":0, "max":0.3}, 'savg')
# Map.addLayer(seg_image_stack.select("nir_shade_2020"), {"min":0, "max":0.3}, 'shade')
Map.addLayer(
    contours.style(color='ffffff', width=1, fillColor='00000000'), 
    {}, 
    'Segment Boundaries'
)
# Map.addLayer(snic.select("clusters"), {}, "seg")
# Map.addLayer(l5_2010_image, vis_params, "l5")

Map

Map(bottom=1775462.0, center=[26.623367961830553, 87.52086730729694], controls=(WidgetControl(options=['positi…

In [ ]:
cluster_stats = seg_image_stack.addBands(config.ROI.reduceToImage(properties=["geoCode"], reducer = ee.Reducer.first())).reduceRegions(collection=contours, scale=target_scale, crs=target_crs, reducer=ee.Reducer.mean())
# stats_df = geemap.ee_to_df(cluster_stats, remove_geom=False)

AttributeError: 'FeatureCollection' object has no attribute 'rename'

In [14]:
target_crs = "EPSG:32645"
target_scale = 30

seeds_test = [3, 4, 5, 6, 7, 8, 9, 10]
connectivity_test = [4, 8]
neighbor_test = [6, 8, 10, 12, 14, 16, 18, 20]
compactness_test = [0.1, 0.2, 0.3, 0.4, 0.5]

paired_seedsNneighborhood = list(zip(seeds_test, neighbor_test))

combination_count=0
for seed, neighborhood in paired_seedsNneighborhood:
    for connectivity in connectivity_test:
        for compactness in compactness_test:
            combination_count += 1
            seeds = ee.Algorithms.Image.Segmentation.seedGrid(seed, "hex").reproject(crs=target_crs, scale=target_scale)
            snic = ee.Algorithms.Image.Segmentation.SNIC(
                image= seg_image_stack.reproject(crs=target_crs, scale=target_scale),
                connectivity= connectivity,
                neighborhoodSize= neighborhood,
                seeds= seeds,
                compactness=compactness
            )
            def extract_and_tag(feature):
                region_loc = feature.get('loc')
                combined_reducer = ee.Reducer.mean().combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
                local_contours = snic.select('clusters').reduceToVectors(
                    geometry=feature.geometry(), 
                    scale=target_scale, 
                    maxPixels=1e13,
                    crs=target_crs
                )
                local_stats = seg_image_stack.reduceRegions(
                    collection=local_contours,
                    scale=target_scale,
                    crs=target_crs,
                    reducer=combined_reducer,
                    tileScale=4
                )
                return local_stats.map(lambda cluster: cluster.set('loc', region_loc))


            cluster_stats = ee.FeatureCollection(config.ROI).map(extract_and_tag).flatten()

            filename = f"s-{seed}_n-{neighborhood}_cn-{connectivity}_cm-{str(compactness)[-1]}"
            
            geemap.ee_export_vector_to_drive(
                collection = cluster_stats,
                description = filename,
                fileFormat = "GeoJSON",
                folder="aal"
            )


Exporting s-3_n-6_cn-4_cm-1... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-4_cm-2... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-4_cm-3... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-4_cm-4... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-4_cm-5... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-8_cm-1... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-8_cm-2... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-8_cm-3... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-8_cm-4... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-3_n-6_cn-8_cm-5... Please check the Task Manager from the JavaScript Code Editor.
Exporting s-4_n-8_cn-4_cm-1... Please check the Task Manager

## Checking suitable segmentation parameters

In [2]:
import os
import glob
import pandas as pd
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran
import numpy as np

In [50]:
geojson_dir = r"outputs\obia_results"
# target_bands = ['eastness_2000', 'eastness_2000',
#        'g_2000', 'g_2000', 'g_2010', 'g_2010',
#        'g_2020', 'g_2020', 'ndvi_2000',
#        'ndvi_2000', 'ndvi_2010', 'ndvi_2010',
#        'ndvi_2020', 'ndvi_2020', 'nir_2000',
#        'nir_2000', 'nir_2010', 'nir_2010',
#        'nir_2020', 'nir_2020', 'nir_savg_2000',
#        'nir_savg_2000', 'nir_savg_2010',
#        'nir_savg_2010', 'nir_savg_2020',
#        'nir_savg_2020', 'nir_shade_2000',
#        'nir_shade_2000', 'nir_shade_2010',
#        'nir_shade_2010', 'nir_shade_2020',
#        'nir_shade_2020', 'northness_2000',
#        'northness_2000', 'r_2000', 'r_2000',
#        'r_2010', 'r_2010', 'r_2020', 'r_2020',
#        'slope_2000', 'slope_2000', 'swir1_2000',
#        'swir1_2000', 'swir1_2010', 'swir1_2010',
#        'swir1_2020', 'swir1_2020', 'swir2_2000',
#        'swir2_2000', 'swir2_2010', 'swir2_2010',
#        'swir2_2020', 'swir2_2020']

# def calculate_metrics(filepath):
#     gdf = gpd.read_file(filepath)

#     if gdf.crs.to_epsg() != 32645:
#         gdf = gdf.to_crs("EPSG:32645")
#     gdf["area"] = gdf.geometry.area
    
#     w = Queen.from_dataframe(gdf, use_index=False, silence_warnings=True)
#     w.transform = "r"

#     alv_list = []
#     mi_list = []
#     for band in target_bands:
#         mean_col = f"{band}_mean"
#         var_col = f"{band}_variance"
#         if mean_col not in gdf.columns or var_col not in gdf.columns:
#             print(f"{mean_col}/{var_col} not in the df")
#             continue
        
#         alv_band = np.sum(gdf[var_col] * gdf["area"]) / np.sum(gdf["area"])
#         alv_list.append(alv_band)

#         mi_band = Moran(gdf[mean_col], w).I
#         mi_list.append(mi_band)
        
#     final_alv = np.nanmean(alv_list)
#     final_mi = np.nanmean(mi_list)

#     return {
#         "filename": os.path.basename(filepath),
#         "ALV": final_alv,
#         "MoranI": final_mi
#     }

In [39]:
calculate_metrics(r"E:\work\Agricultural Land Abandonment\code\outputs\obia_results\s-10_n-20_cn-4_cm-5.geojson")

{'filename': 's-10_n-20_cn-4_cm-5.geojson',
 'ALV': np.float64(0.0012914445300355056),
 'MoranI': np.float64(0.5889498606040466)}

In [52]:
len(glob.glob(os.path.join(geojson_dir, "*.geojson")))

80